In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from accelerate import Accelerator
from accelerate.utils import set_seed
import subprocess
import os
import time

torch.set_float32_matmul_precision('high')

# 自动配置代理（针对特定环境）
result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

def train(compile=True):
    set_seed(42)
    accelerator = Accelerator(mixed_precision='bf16')
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=16)

    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=16)
    model = torchvision.models.resnet50(num_classes=10)

    if compile:
        s_compile_time = time.time()
        model = torch.compile(model, mode="reduce-overhead")
        accelerator.print(f"Compile Time: {time.time() - s_compile_time:.2f}s")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-5)

    model, optimizer, train_loader, test_loader = accelerator.prepare(
        model, optimizer, train_loader, test_loader
    )
    for epoch in range(100):
        # --- 训练阶段 ---
        model.train()
        train_correct = 0
        train_samples = 0
        s_train_time = time.time()

        batch_time = []
        for batch_idx, (data, target) in enumerate(train_loader):
            s_batch_time = time.time()
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            accelerator.backward(loss)
            optimizer.step()

            # 统计训练准确率
            predictions = output.argmax(dim=-1)
            all_preds, all_targets = accelerator.gather_for_metrics((predictions, target))
            train_correct += (all_preds == all_targets).sum().item()
            train_samples += all_targets.size(0)
            batch_time.append(time.time() - s_batch_time)

        train_acc = 100.0 * train_correct / train_samples

        model.eval()
        test_correct = 0
        test_samples = 0

        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                predictions = output.argmax(dim=-1)
                all_preds, all_targets = accelerator.gather_for_metrics((predictions, target))
                test_correct += (all_preds == all_targets).sum().item()
                test_samples += all_targets.size(0)
        test_acc = 100.0 * test_correct / test_samples
        if epoch % 5== 0:
            accelerator.print(
                f"Epoch {epoch:02d} | Train Time: {time.time() - s_train_time:.2f}s | Batch Time: {sum(batch_time)/len(batch_time)}"
                f"Train ACC: {train_acc:.2f}% | Test ACC: {test_acc:.2f}%"
            )

if __name__ == "__main__":
    # train(compile= True)
    train(compile= False)

Epoch 00 | Train Time: 5.71s | Batch Time: 0.05050786174073511Train ACC: 11.29% | Test ACC: 11.41%
Epoch 05 | Train Time: 5.25s | Batch Time: 0.047670155155415436Train ACC: 15.84% | Test ACC: 16.45%
Epoch 10 | Train Time: 6.11s | Batch Time: 0.04854408575564015Train ACC: 18.56% | Test ACC: 19.64%
Epoch 15 | Train Time: 6.33s | Batch Time: 0.04916523427379375Train ACC: 20.40% | Test ACC: 22.29%
Epoch 20 | Train Time: 6.09s | Batch Time: 0.051163284146055886Train ACC: 22.42% | Test ACC: 23.61%
Epoch 25 | Train Time: 5.79s | Batch Time: 0.04896752201780981Train ACC: 23.90% | Test ACC: 25.22%
Epoch 30 | Train Time: 5.78s | Batch Time: 0.05085692113759566Train ACC: 25.14% | Test ACC: 26.61%
Epoch 35 | Train Time: 5.12s | Batch Time: 0.04880750422575036Train ACC: 26.75% | Test ACC: 28.13%
Epoch 40 | Train Time: 5.38s | Batch Time: 0.04720686406505351Train ACC: 27.84% | Test ACC: 29.21%
Epoch 45 | Train Time: 5.10s | Batch Time: 0.047691140856061666Train ACC: 28.90% | Test ACC: 30.45%
Epoch 5

In [ ]:
Compile Time: 0.89s
Epoch 00 | Train Time: 12.17s | Batch Time: 0.14764126466245067Train ACC: 11.19% | Test ACC: 11.89%
Epoch 05 | Train Time: 5.72s | Batch Time: 0.046339944917328506Train ACC: 15.49% | Test ACC: 15.54%
Epoch 10 | Train Time: 5.71s | Batch Time: 0.04499245176509935Train ACC: 18.26% | Test ACC: 19.50%
Epoch 15 | Train Time: 5.27s | Batch Time: 0.047159457693294604Train ACC: 20.47% | Test ACC: 21.51%
Epoch 20 | Train Time: 5.33s | Batch Time: 0.044611001501278Train ACC: 22.09% | Test ACC: 23.85%
Epoch 25 | Train Time: 5.30s | Batch Time: 0.04631211319748236Train ACC: 23.77% | Test ACC: 24.83%
Epoch 30 | Train Time: 5.57s | Batch Time: 0.04546777082949269Train ACC: 25.45% | Test ACC: 27.15%
Epoch 35 | Train Time: 6.04s | Batch Time: 0.04685194638310647Train ACC: 26.64% | Test ACC: 27.61%
Epoch 40 | Train Time: 5.78s | Batch Time: 0.04502055109763632Train ACC: 27.74% | Test ACC: 29.68%
Epoch 45 | Train Time: 5.89s | Batch Time: 0.04582952966495436Train ACC: 29.10% | Test ACC: 30.91%
Epoch 50 | Train Time: 5.03s | Batch Time: 0.046522539489123285Train ACC: 30.10% | Test ACC: 31.79%
Epoch 55 | Train Time: 5.26s | Batch Time: 0.045492488510754645Train ACC: 31.21% | Test ACC: 33.37%
Epoch 60 | Train Time: 5.41s | Batch Time: 0.04656180556939573Train ACC: 32.18% | Test ACC: 34.68%
Epoch 65 | Train Time: 5.60s | Batch Time: 0.04664086322395169Train ACC: 33.27% | Test ACC: 35.17%
Epoch 70 | Train Time: 5.81s | Batch Time: 0.04666753691069934Train ACC: 34.34% | Test ACC: 35.94%
Epoch 75 | Train Time: 5.69s | Batch Time: 0.04741260956744758Train ACC: 35.30% | Test ACC: 37.51%
Epoch 80 | Train Time: 5.99s | Batch Time: 0.049882051896075814Train ACC: 35.83% | Test ACC: 38.80%
Epoch 85 | Train Time: 5.60s | Batch Time: 0.04556511859504544Train ACC: 37.01% | Test ACC: 39.24%
Epoch 90 | Train Time: 5.70s | Batch Time: 0.04652732245776118Train ACC: 37.74% | Test ACC: 40.30%
Epoch 95 | Train Time: 5.66s | Batch Time: 0.04582754933104223Train ACC: 38.30% | Test ACC: 41.03%
